In [1]:
from pydantic import BaseModel
class User(BaseModel):
    id:int
    name:str
    is_active:bool

input_data={'id':1,'name':'Nithya Sai','is_active':True}

user1=User(**input_data)


In [2]:
user1

User(id=1, name='Nithya Sai', is_active=True)

In [4]:
#nested model
class Address(BaseModel):
    street: str
    city: str
    country:str
class User(BaseModel):
    id:int
    name:str
    address:Address

address=Address(street="Gandhi street",city="Hyderabad",country="India")

user_data={
    "id":"123",
    "name":"raju",
    "address":address

}

user=User(**user_data)
print(user)
print(user.address
      )

id=123 name='raju' address=Address(street='Gandhi street', city='Hyderabad', country='India')
street='Gandhi street' city='Hyderabad' country='India'


In [5]:
class Product(BaseModel):
    id:int
    name:str
    price:float
    in_stock:bool

prod1=Product(id=1,name="Chair",price=29.09,in_stock=True)
prod2=Product(id=2,name="bed")

ValidationError: 2 validation errors for Product
price
  Field required [type=missing, input_value={'id': 2, 'name': 'bed'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
in_stock
  Field required [type=missing, input_value={'id': 2, 'name': 'bed'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

In [ ]:
#self referencing
from typing import List,Optional
from pydantic import BaseModel

class Comment(BaseModel):
    id:int
    content:str
    replies: Optional[List['Comment']]=None

#in the above code we pass the Class name as string literal to avoid the run time error. 
# and below code will tell the interpreter to use the Class as the filler 
Comment.model_rebuild()

comment=Comment(
    id=1,
    content="Hey! how do you do?",
    replies=[
        Comment(id=2,content="I am good"),
        Comment(id=3,content="Hey dude",replies=[Comment(id=4,content="I am dude")])
    ]
)
comment

Comment(id=1, content='Hey! how do you do?', replies=[Comment(id=2, content='I am good', replies=None), Comment(id=3, content='Hey dude', replies=[Comment(id=4, content='I am dude', replies=None)])])

In [3]:
#Field validators and Model validator
from pydantic import BaseModel,field_validator,model_validator

class User(BaseModel):
    username:str

    @field_validator('username')
    def username_length(cls,v):
        if len(v)<4:
            raise ValueError("usernamemust be atleast 4 characters")
        return v
    
class SignupData(BaseModel):
    password:str
    confirm_password:str

    @model_validator(mode='after')
    def password_match(cls,values):
        if values.password !=values.confirm_password:
            raise ValueError("Password do not match")
        return values


/tmp/ipykernel_7470/1867482441.py:18: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.13/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  def password_match(cls,values):


In [5]:
user=User(username='opi')

ValidationError: 1 validation error for User
username
  Value error, usernamemust be atleast 4 characters [type=value_error, input_value='opi', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [6]:
passw_ord=SignupData(password='loiu',confirm_password='ksjj')

ValidationError: 1 validation error for SignupData
  Value error, Password do not match [type=value_error, input_value={'password': 'loiu', 'confirm_password': 'ksjj'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [9]:
#typing module
from typing import List,Dict,Optional

class Cart(BaseModel):
    user_id:int
    items:List[str]
    quantities: Dict[str,int]

class BlogPosts(BaseModel):
    title:str
    content:str
    image_url:Optional[str]=None

cart_data={
    'user_id':"123",
    'items':['Laptop','Chair','Table'],
    'quantities':{
        'laptop':20,'Chair':12,'Table':13
    }
}

cart=Cart(**cart_data)

In [10]:
cart

Cart(user_id=123, items=['Laptop', 'Chair', 'Table'], quantities={'laptop': 20, 'Chair': 12, 'Table': 13})

In [3]:
#working with Field 
#every attribute is a field,but we can use conditions also

from pydantic import Field,BaseModel
from typing import Optional
import re

class Employee(BaseModel):
    id:int
    name:str = Field(...,min_length=3,max_length=25,description="Employee Name",examples="Rajesh")
    department:Optional[str] ='general'
    salary:float = Field(...,ge=10000)

class User_Fields(BaseModel):
    email:str= Field(...,pattern=r"^[a-zA-Z0-9_]{4,15}$")
    phone:str =Field(...,pattern=r'')
    age:int = Field(...,ge=0,le=120,description="Age in Years")
    discount: float=Field(...,ge=0,le=100,description="Discount percentage")




In [4]:
emp=Employee(id=1,name="raj",salary=12092)


In [5]:
emp

Employee(id=1, name='raj', department='general', salary=12092.0)

In [6]:
user=User_Fields(email='',phone=' ',age=10,discount=19)

ValidationError: 1 validation error for User_Fields
email
  String should match pattern '^[a-zA-Z0-9_]{4,15}$' [type=string_pattern_mismatch, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_pattern_mismatch

In [17]:
user

User_Fields(email=' ', phone=' ', age=10, discount=19.0)

In [23]:
#check regex pattern

In [7]:
from pydantic import BaseModel,computed_field,Field

class Product(BaseModel):
    price:float
    quantity:int
    @computed_field
    @property
    def total_price(self)->float:
        return self.price*self.quantity
    
class Booking(BaseModel):
    user_id:int
    room_id:int
    nights: int =Field(...,ge=1)
    rate_per_night:float

    @computed_field
    @property
    def total_amount(self)-> float:
        return self.nights*self.rate_per_night
    


In [8]:
booking=Booking(user_id=123,
                room_id=1,
                nights=3,
                rate_per_night=121)

In [ ]:
type(booking.model_dump())


dict

In [12]:
type(booking.model_dump_json())

str

In [ ]:
booking.model_dump_json() ##computed field is seen as attribute

'{"user_id":123,"room_id":1,"nights":3,"rate_per_night":121.0,"total_amount":363.0}'

In [14]:
#field_validator and model_validator

from pydantic import BaseModel,field_validator,model_validator
from datetime import datetime

class User(BaseModel):
    email:str

    @field_validator('email')
    def normalize_email(cls,v):
        return v.lower().strip()


In [15]:
user=User(email="Nithyasaitejareddy99@gmail.com")

In [16]:
user.email

'nithyasaitejareddy99@gmail.com'

In [20]:
class Product(BaseModel):
    price:float
    @field_validator('price',mode='before')
    def parse_price(cls,v):
        if isinstance(v,str):
            return float(v.replace('$',''))
        return v
    


In [21]:
product1=Product(price='24$')
product1.price

24.0

In [32]:
class DateRange(BaseModel):
    start_date:datetime
    end_date:datetime

    @model_validator(mode='after')
    def validate_date_range(self):
        if self.end_date<=self.start_date:
            raise ValueError("end date must be after start_date")
        return self
        
    


In [33]:
date_values=DateRange(start_date='2026-05-19T22:31:57',end_date='2026-05-19T22:31:58')

In [34]:
date_values

DateRange(start_date=datetime.datetime(2026, 5, 19, 22, 31, 57), end_date=datetime.datetime(2026, 5, 19, 22, 31, 58))

In [35]:
#nested models
class Address(BaseModel):
    street:str
    city:str
    postal_code:str

class Company(BaseModel):
    name: str
    address:Address

class Employee(BaseModel):
    id:int
    name:str
    company:Optional[Company]=None
    

In [37]:
from typing import Union,List
class textContent(BaseModel):
    type:str="text"
    content:str

class ImageContent(BaseModel):
    type:str="image"
    url:str
    alt_text:str

class Article(BaseModel):
    title:str
    sections: List[Union[textContent,ImageContent]]

In [38]:
#serialization
from pydantic import BaseModel,ConfigDict
from typing import List
from datetime import datetime

class Address(BaseModel):
    street:str
    city:str
    zip_code:str

class User(BaseModel):
    id:int
    name:str
    email:str
    is_active:bool=True
    createdAt:datetime
    address:Address
    tags:List[str]=[]

    model_config=ConfigDict(
        json_encoders={datetime: lambda v:v.strftime('%d-%m-%Y %H:%M:%S')}

    )


    

In [40]:
user=User(
    id=1,
    name="Nithya Sai",
    email="nithyasaitejareddy.gmail.com",
    createdAt=datetime(2025,3,19,14,29),
    address=Address(street="Nehru Rd",city="jaisalmer",zip_code="193873"),
    
)

python_dict=user.model_dump()
python_dict

{'id': 1,
 'name': 'Nithya Sai',
 'email': 'nithyasaitejareddy.gmail.com',
 'is_active': True,
 'createdAt': datetime.datetime(2025, 3, 19, 14, 29),
 'address': {'street': 'Nehru Rd', 'city': 'jaisalmer', 'zip_code': '193873'},
 'tags': []}

In [41]:
json_value=user.model_dump_json()
json_value

'{"id":1,"name":"Nithya Sai","email":"nithyasaitejareddy.gmail.com","is_active":true,"createdAt":"19-03-2025 14:29:00","address":{"street":"Nehru Rd","city":"jaisalmer","zip_code":"193873"},"tags":[]}'